> ⏱ ~20 min

# Round 1 — Baseline

You send the brand-new MAF agent to a panel of trip reviewers. The agent has only generic helper instructions — no travel policy, no catalog facts, no specialist tools. The scores you record here are the **baseline** that every later round is compared against.

An **evaluator** is a checker that scores one aspect of an answer (relevance, coherence, groundedness, and so on). A **baseline** is the first score you keep, so you can later say "round 2 improved relevance by X points" instead of "the new version feels better."

## What you will do

1. Build a single MAF `Agent` with generic instructions and no tools.
2. Wrap it as a synchronous callable that `azure-ai-evaluation.evaluate(...)` can drive.
3. Run the standard built-in evaluators against the 20-prompt workshop dataset.
4. Save the metrics to `eval-results/round1-baseline-*`, ready to compare against rounds 2-4.

Keeping the dataset and the evaluator suite constant across rounds is what makes the comparison fair — only the agent changes.

---

In [ ]:
# Suppress experimental/deprecation warnings to keep learning output clean.
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Locate this lab folder portably (do not hardcode the repo name; learners may fork+rename).
LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab-1-foundry-maf':
    for _p in (LAB_DIR, *LAB_DIR.parents):
        _candidate = _p / 'cohere' / 'lab-1-foundry-maf'
        if _candidate.exists():
            LAB_DIR = _candidate
            break
COHERE_DIR = LAB_DIR.parent
load_dotenv(COHERE_DIR / '.env')
sys.path.insert(0, str(LAB_DIR))

PROJECT_ENDPOINT = os.environ['FOUNDRY_PROJECT_ENDPOINT']
DEPLOYMENT = os.getenv('COMMAND_A_DEPLOYMENT', 'command-a')
DATASET = LAB_DIR / '04-eval-dataset.jsonl'
print('Dataset rows:', sum(1 for _ in DATASET.open(encoding='utf-8')))

## 1. Build the baseline agent

A baseline agent has nothing extra: no policy in its instructions, no catalog facts, no booking tools. It just has the model. This is on purpose — you want to see what the raw model can do before you start grounding it.

In [ ]:
import asyncio, threading
from agents import make_chat_client, build_baseline_agent, BASELINE_INSTRUCTIONS

client = make_chat_client()
agent = build_baseline_agent(client, BASELINE_INSTRUCTIONS)

# A single background thread owns one event loop. Both the kernel main
# thread (sanity call) and promptflow's worker threads (evaluate runs) hand
# their coroutines to this loop via run_coroutine_threadsafe, so MAF's
# telemetry ContextVar tokens are always set and reset inside the same task
# context.
_runner_loop = asyncio.new_event_loop()
_runner_thread = threading.Thread(
    target=_runner_loop.run_forever, name='eval-loop', daemon=True
)
_runner_thread.start()

async def _invoke(query: str):
    return await agent.run(query)

def target(query: str, **kwargs: object) -> dict:
    """Run the MAF agent for one dataset row and return the response.

    Per-row failures are swallowed so a single bad row never fails the whole
    batch and blocks evaluators from running on the rest of the dataset.
    """
    try:
        fut = asyncio.run_coroutine_threadsafe(_invoke(query), _runner_loop)
        response = fut.result()
        return {'response': response.text or '', 'query': query, 'error': ''}
    except Exception as exc:
        return {
            'response': f'[agent call failed: {type(exc).__name__}: {exc}]',
            'query': query,
            'error': f'{type(exc).__name__}: {exc}',
        }

# Quick sanity call before the long evaluate() run.
print(target('Find an economy flight from SFO to NYC on 2026-07-10 returning 2026-07-13 under $400.')['response'][:400])

## 2. Build the evaluator panel

Different versions of `azure-ai-evaluation` expose different evaluators. The helper below picks up only the ones present in your install, so it does not break across SDK versions. Each evaluator uses Cohere `command-a` as the judge model.

In [ ]:
from azure.identity import DefaultAzureCredential

# Cohere `command-a` is reachable on the account-level OpenAI-compatible path
# (/openai/v1/chat/completions), NOT on the legacy Azure OpenAI deployment path
# (/openai/deployments/{name}/chat/completions). So the evaluators' judge model
# must be configured as `type: openai` with base_url pointing at /openai/v1.
# We pass an Entra access token as the api_key (valid ~1 hour, well over the
# few minutes an eval run takes).
credential = DefaultAzureCredential()
_judge_token = credential.get_token('https://cognitiveservices.azure.com/.default').token
model_config = {
    'type': 'openai',
    'base_url': os.environ['AZURE_AI_ENDPOINT'].rstrip('/') + '/openai/v1',
    'model': DEPLOYMENT,
    'api_key': _judge_token,
}

def build_evaluators():
    import azure.ai.evaluation as evaluation
    evaluators = {}
    for name in ['RelevanceEvaluator', 'CoherenceEvaluator', 'FluencyEvaluator', 'GroundednessEvaluator']:
        cls = getattr(evaluation, name, None)
        if cls is None:
            continue
        try:
            evaluators[name.replace('Evaluator', '').lower()] = cls(model_config)
        except Exception as exc:
            # Preview versions sometimes change the config schema; skip rather than break the run.
            print(f'Skipping {name}: {exc}')
    # ToolCallAccuracyEvaluator is intentionally skipped: it requires
    # `tool_definitions` and `tool_calls` columns that the local target does
    # not emit. Add it back once those are captured per row.
    for name in ['IntentResolutionEvaluator', 'TaskAdherenceEvaluator']:
        cls = getattr(evaluation, name, None)
        if cls is None:
            continue
        try:
            evaluators[name.replace('Evaluator', '').lower()] = cls(model_config)
        except Exception as exc:
            print(f'Skipping {name}: {exc}')
    return evaluators

evaluators = build_evaluators()
print('Active evaluators:', sorted(evaluators))

## 3. Run the baseline evaluation

`evaluate(...)` walks every dataset row, calls `target(query=row['query'])`, then hands each result to every evaluator. The aggregate metrics come back in `result.metrics`. The run is also saved locally and (best-effort) uploaded to your Foundry project so all four rounds appear in the same Evaluations list.

In [ ]:
from datetime import datetime
from azure.ai.evaluation import evaluate
import json as _json
import pandas as pd

# Cap parallel target + evaluator calls so the 100-RPM command-a deployment
# is not bursted past its rate limit. With 6 judge evaluators per row, even
# 4 workers will trigger 429s; 2 keeps the burst comfortably under 100 RPM.
# Set this BEFORE evaluate() builds its batch engine. Use plain assignment
# (not setdefault) so re-running the cell without a kernel restart picks up
# the new value if you ever tune it.
os.environ['PF_WORKER_COUNT'] = '2'

# Hub-less Microsoft Foundry projects are identified by their project endpoint;
# the legacy AzureAIProject(subscription, resource_group, project_name) triple
# resolves to an ML workspace that does not exist for these projects.
azure_ai_project = PROJECT_ENDPOINT

RESULTS_DIR = LAB_DIR / 'eval-results'
RESULTS_DIR.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
output_path = RESULTS_DIR / f'round1-baseline-{timestamp}.json'

def _run_evaluation(project):
    return evaluate(
        data=str(DATASET),
        target=target,
        evaluators=evaluators,
        azure_ai_project=project,
        evaluation_name='round1-baseline',
        fail_on_evaluator_errors=False,
        tags={'lab': 'cohere-lab-1-foundry-maf', 'round': 'round1-baseline'},
        output_path=str(output_path),
    )

try:
    result = _run_evaluation(azure_ai_project)
    uploaded = True
except Exception as exc:
    print(f'Portal upload failed ({type(exc).__name__}: {exc}); rerunning local-only.')
    result = _run_evaluation(None)
    uploaded = False

metrics = getattr(result, 'metrics', None) or (result.get('metrics') if isinstance(result, dict) else {})
rows = getattr(result, 'rows', None) or (result.get('rows') if isinstance(result, dict) else [])

rows_df = pd.DataFrame(rows)
rows_path = RESULTS_DIR / f'round1-baseline-{timestamp}-rows.csv'
metrics_path = RESULTS_DIR / f'round1-baseline-{timestamp}-metrics.json'
rows_df.to_csv(rows_path, index=False)
metrics_path.write_text(_json.dumps(metrics, indent=2, sort_keys=True))

print(f'Portal upload: {"yes" if uploaded else "no (local-only)"}')
print(f'Full result : {output_path}')
print(f'Per-row CSV : {rows_path}')
print(f'Metrics JSON: {metrics_path}')
pd.DataFrame(sorted(metrics.items()), columns=['metric', 'value'])

## What you confirmed

- A bare MAF agent (no policy, no tools) is a working baseline.
- The same evaluators that will score later rounds work against this agent without any change to the dataset.
- The metrics file in `eval-results/` is your reference point.

Next: **`04-eval-round2-grounded.ipynb`** adds the travel policy and catalog summaries to the agent's instructions and reruns the same evaluators.